# Minimale Microsoft-Anmeldung für Backstage

Dieses Notebook beschreibt die einfachste Integration einer Microsoft-/Microsoft-365-Anmeldung in Backstage. Kubernetes wird nicht verwendet.


## 1. Anwendung in Microsoft Entra ID registrieren

Im **Microsoft Entra Admin Center**:

1. **App registrations** öffnen.
2. **New registration** auswählen.
3. Als Namen beispielsweise `Backstage` verwenden.
4. Für eine interne Umgebung **Single tenant** auswählen.
5. Die Anwendung registrieren.

Unter **Authentication → Add a platform → Web** folgende Redirect URI eintragen:

```text
http://localhost:7007/api/auth/microsoft/handler/frame
```

Unter **Certificates & secrets** ein neues Client Secret erstellen.

Folgende Werte werden benötigt:

- Application beziehungsweise Client ID
- Client Secret
- Directory beziehungsweise Tenant ID

Als delegierte Microsoft-Graph-Berechtigungen werden typischerweise folgende Berechtigungen verwendet:

- `email`
- `offline_access`
- `openid`
- `profile`
- `User.Read`


## 2. Microsoft-Backend-Modul installieren

Im Hauptverzeichnis der Backstage-Installation ausführen:

In [ ]:
%%bash
yarn --cwd packages/backend add @backstage/plugin-auth-backend-module-microsoft-provider

Anschliessend in `packages/backend/src/index.ts` ergänzen:

```typescript
backend.add(import('@backstage/plugin-auth-backend'));
backend.add(
  import('@backstage/plugin-auth-backend-module-microsoft-provider'),
);
```


## 3. `app-config.yaml` konfigurieren

Die folgende Konfiguration ergänzt den Microsoft-Provider für die Entwicklungsumgebung:

```yaml
auth:
  environment: development
  providers:
    microsoft:
      development:
        clientId: ${MICROSOFT_CLIENT_ID}
        clientSecret: ${MICROSOFT_CLIENT_SECRET}
        tenantId: ${AZURE_TENANT_ID}
        signIn:
          resolvers:
            - resolver: emailMatchingUserEntityProfileEmail
```

Der Resolver verknüpft die vom Microsoft-Konto gelieferte E-Mail-Adresse mit dem Feld `spec.profile.email` einer Backstage-User-Entity.


## 4. Backstage-Benutzer erfassen

Beispielsweise in `examples/users.yaml`:

```yaml
apiVersion: backstage.io/v1alpha1
kind: User
metadata:
  name: marcel
spec:
  profile:
    displayName: Marcel
    email: marcel@example.ch
  memberOf: []
```

Die E-Mail-Adresse muss mit der E-Mail-Adresse des Microsoft-Kontos übereinstimmen.

Falls die Datei noch nicht im Catalog registriert ist, kann sie in `app-config.yaml` eingebunden werden:

```yaml
catalog:
  locations:
    - type: file
      target: ../../examples/users.yaml
```


## Ergebnis

Nach diesen vier Schritten sind die Microsoft-Anwendung, das Backstage-Backend-Modul, der Authentifizierungsprovider und die Benutzerzuordnung vorbereitet.
